In [ ]:
!pip install open3d --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.1 MB/s eta 0:00:00


In [ ]:
!pip install -U fvcore --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Core
import os
import json
import glob
import numpy as np
from pathlib import Path

# Point cloud
import open3d as o3d          # read .ply files

# Image
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# FLOP counting
from fvcore.nn import FlopCountAnalysis   # preferred
# import thop                             # alternative

# Profiling & memory
from torch.profiler import profile, record_function, ProfilerActivity
import torch.cuda

# Numerics & visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd

# Install if not present
# !pip install open3d fvcore

Mounted at /content/drive


In [ ]:
import os

# Search for the file in Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'train1' in f:
            print(os.path.join(root, f))

/content/drive/MyDrive/train1.gz
/content/drive/MyDrive/argoverse-tracking_train1.zip


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/argoverse-tracking_train1.zip"
extract_path = "/content/argoverse_data"

# Create extraction folder
os.makedirs(extract_path, exist_ok=True)

# Extract zip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

Extraction complete!


In [ ]:
import os

for root, dirs, files in os.walk('/content/argoverse_data/'):
    uuid_dirs = [d for d in dirs if len(d) == 36 and d.count('-') == 4]
    if uuid_dirs:
        print(f"ROOT = '{root}'")
        print(f"Total sequences: {len(uuid_dirs)}")
        print(f"Example: {uuid_dirs[0]}")
        break

ROOT = '/content/argoverse_data/'
Total sequences: 10
Example: dcdcd8b3-0ba1-3218-b2ea-7bb965aad3f0


In [ ]:
import random

DATASET_ROOT  = '/content/argoverse_data/'
all_sequences = sorted([
    os.path.join(DATASET_ROOT, d)
    for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
])

print(f"Using {len(all_sequences)} sequences total")

random.seed(42)
random.shuffle(all_sequences)

split           = int(0.7 * len(all_sequences))   # 70/30 split
train_sequences = all_sequences[:split]            # ~10-11 sequences
val_sequences   = all_sequences[split:]            # ~4-5 sequences

print(f"Train: {len(train_sequences)} | Val: {len(val_sequences)}")

Using 10 sequences total
Train: 7 | Val: 3


In [ ]:
import os

seq = '/content/argoverse_data/25952736-2595-2595-2595-225953853440'

CAMERAS = [
    'ring_front_center',
    'ring_front_left',
    'ring_front_right',
    'ring_rear_left',
    'ring_rear_right',
    'ring_side_left',
    'ring_side_right',
    'stereo_front_left',
    'stereo_front_right',
]

lidar_path = os.path.join(seq, 'lidar')
annot_path = os.path.join(seq, 'per_sweep_annotations_amodal')

lidar_files = sorted(os.listdir(lidar_path))[:5] if os.path.exists(lidar_path) else []
annot_files = sorted(os.listdir(annot_path))[:3] if os.path.exists(annot_path) else []

print("LiDAR files:", lidar_files)
print("Annot files:", annot_files)

for cam in CAMERAS:
    cam_path = os.path.join(seq, cam)
    if os.path.exists(cam_path):
        files = sorted(os.listdir(cam_path))[:3]
        print(f"{cam}: {files}")
    else:
        print(f"{cam}: NOT FOUND")

LiDAR files: ['PC_315966391619461000.ply', 'PC_315966391719658000.ply', 'PC_315966391819191000.ply', 'PC_315966391919387000.ply', 'PC_315966392019566000.ply']
Annot files: ['tracked_object_labels_315966391619461000.json', 'tracked_object_labels_315966391719658000.json', 'tracked_object_labels_315966391819191000.json']
ring_front_center: ['ring_front_center_315966391606850384.jpg', 'ring_front_center_315966391640150776.jpg', 'ring_front_center_315966391673451088.jpg']
ring_front_left: ['ring_front_left_315966391606853040.jpg', 'ring_front_left_315966391640153160.jpg', 'ring_front_left_315966391673453400.jpg']
ring_front_right: ['ring_front_right_315966391606853448.jpg', 'ring_front_right_315966391640153624.jpg', 'ring_front_right_315966391673453800.jpg']
ring_rear_left: ['ring_rear_left_315966391606857320.jpg', 'ring_rear_left_315966391640157304.jpg', 'ring_rear_left_315966391673457288.jpg']
ring_rear_right: ['ring_rear_right_315966391606860816.jpg', 'ring_rear_right_315966391640160512.

In [ ]:
def extract_ts(filepath):
    return int(Path(filepath).stem.split('_')[-1])

def get_synced_pairs(lidar_dir, cam_dir, max_dt_ns=50_000_000):
    lidar_files = sorted(glob.glob(lidar_dir + "*.ply"))
    cam_files   = sorted(glob.glob(cam_dir   + "*.jpg"))

    if not lidar_files or not cam_files:
        print(f"  ⚠️  Empty: lidar={len(lidar_files)}, cam={len(cam_files)}")
        return []

    lidar_ts = [(extract_ts(f), f) for f in lidar_files]
    cam_ts   = [(extract_ts(f), f) for f in cam_files]
    cam_arr  = np.array([t for t, _ in cam_ts])

    pairs = []
    for l_ts, l_path in lidar_ts:
        idx          = np.argmin(np.abs(cam_arr - l_ts))
        c_ts, c_path = cam_ts[idx]
        dt           = abs(l_ts - c_ts)
        if dt <= max_dt_ns:
            pairs.append({"lidar_path": l_path, "cam_path": c_path,
                          "lidar_ts": l_ts, "cam_ts": c_ts, "dt_ms": dt/1e6})
    return pairs

def load_annotations(json_path):
    with open(json_path) as f:
        data = json.load(f)
    return [{
        "label":   obj["label_class"],
        "center":  obj["center"],
        "dims":    [obj["length"], obj["width"], obj["height"]],
        "heading": obj.get("rotation", {}).get("z", 0.0),
        "uuid":    obj["track_label_uuid"],
    } for obj in data]

def load_image(img_path, size=(640, 640)):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size)
    return T.ToTensor()(img)

In [ ]:
import os
import glob
import json
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

from tqdm import tqdm

In [ ]:
class ArgoverseDataset(Dataset):
    def __init__(self, seq_roots, cam_name="ring_front_center"):
        self.samples = []
        for seq in seq_roots:
            lidar_dir = os.path.join(seq, "lidar/")
            annot_dir = os.path.join(seq, "per_sweep_annotations_amodal/")
            cam_dir   = os.path.join(seq, f"{cam_name}/")

            if not all(os.path.exists(p) for p in [lidar_dir, annot_dir, cam_dir]):
                print(f"  ⚠️  Skipping {os.path.basename(seq)}")
                continue

            pairs = get_synced_pairs(lidar_dir, cam_dir)
            for p in pairs:
                lidar_ts_str = str(extract_ts(p["lidar_path"]))
                self.samples.append({
                    "img":   p["cam_path"],
                    "lidar": p["lidar_path"],
                    "annot": os.path.join(annot_dir, f"tracked_object_labels_{lidar_ts_str}.json"),
                    "seq":   os.path.basename(seq),
                    "ts":    lidar_ts_str,
                })
            print(f"  ✅ {os.path.basename(seq)}: {len(pairs)} frames")
        print(f"\nTotal frames: {len(self.samples)}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = load_image(s["img"])
        try:    annots = load_annotations(s["annot"])
        except: annots = []
        return image, annots, s["seq"], s["ts"]


train_dataset = ArgoverseDataset(train_sequences, cam_name="ring_front_center")
val_dataset   = ArgoverseDataset(val_sequences,   cam_name="ring_front_center")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=2, collate_fn=lambda x: x)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False,
                          num_workers=2, collate_fn=lambda x: x)

print(f"\nTrain: {len(train_dataset)} frames, {len(train_loader)} batches")
print(f"Val  : {len(val_dataset)} frames, {len(val_loader)} batches")

  ✅ dcdcd8b3-0ba1-3218-b2ea-7bb965aad3f0: 155 frames
  ✅ 64c12551-adb9-36e3-a0c1-e43a0e9f3845: 155 frames
  ✅ 3d20ae25-5b29-320d-8bae-f03e9dc177b9: 156 frames
  ✅ e17eed4f-3ffd-3532-ab89-41a3f24cf226: 156 frames
  ✅ 84c35ea7-1a99-3a0c-a3ea-c5915d68acbc: 293 frames
  ✅ a073e840-6319-3f0b-843e-f6dccdcc7b77: 299 frames
  ✅ f3fb839e-0aa2-342b-81c3-312b80be44f9: 156 frames

Total frames: 1370
  ⚠️  Skipping 6f153f9c-edc5-389f-ac6f-40705c30d97e
  ✅ 25952736-2595-2595-2595-225953853440: 147 frames
  ✅ 273c1883-673a-36bf-b124-88311b1a80be: 146 frames

Total frames: 293

Train: 1370 frames, 343 batches
Val  : 293 frames, 74 batches


In [ ]:
import timm

class FPN(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        self.lateral = nn.ModuleList([nn.Conv2d(c, out_channels, 1) for c in in_channels_list])
        self.output  = nn.ModuleList([nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels_list])

    def forward(self, features):
        laterals = [l(f) for l, f in zip(self.lateral, features)]
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i-1] += nn.functional.interpolate(laterals[i], size=laterals[i-1].shape[-2:], mode='nearest')
        return [o(l) for o, l in zip(self.output, laterals)]


class DetectionHead3D(nn.Module):
    def __init__(self, in_channels=256, num_classes=15, num_anchors=9):
        super().__init__()
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1), nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * num_classes, 1))
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1), nn.ReLU(),
            nn.Conv2d(in_channels, num_anchors * 7, 1))

    def forward(self, fpn_features):
        cls_outs, reg_outs = [], []
        for feat in fpn_features:
            cls_outs.append(self.cls_head(feat))
            reg_outs.append(self.reg_head(feat))
        return cls_outs, reg_outs


BACKBONE_CONFIG = {
    (18,  0.5): "resnet18",
    (18,  1.0): "resnet18",
    (18,  2.0): "resnet18",
    (34,  0.5): "resnet34",
    (34,  1.0): "resnet34",
    (34,  2.0): "resnet34",
    (50,  0.5): "resnet50",
    (50,  1.0): "resnet50",
    (50,  2.0): "wide_resnet50_2",
    (101, 0.5): "resnet101",
    (101, 1.0): "resnet101",
    (101, 2.0): "wide_resnet101_2",
}

class Detector3D(nn.Module):
    def __init__(self, depth=50, width_mult=1.0, num_classes=15):
        super().__init__()
        model_name    = BACKBONE_CONFIG[(depth, width_mult)]
        self.backbone = timm.create_model(model_name, pretrained=True,
                                          features_only=True, out_indices=(2, 3, 4))
        fpn_out_ch = {0.5: 128, 1.0: 256, 2.0: 512}[width_mult]
        with torch.no_grad():
            feats = self.backbone(torch.zeros(1, 3, 640, 640))
        in_ch = [f.shape[1] for f in feats]
        self.fpn  = FPN(in_channels_list=in_ch, out_channels=fpn_out_ch)
        self.head = DetectionHead3D(in_channels=fpn_out_ch, num_classes=num_classes)

    def forward(self, x):
        features  = self.backbone(x)
        fpn_feats = self.fpn(features)
        return self.head(fpn_feats)

In [ ]:
model = Detector3D(depth=50, width_mult=1.0, num_classes=15).cuda()
dummy_input = torch.randn(2, 3, 640, 640).cuda()
cls_out, reg_out = model(dummy_input)
print("CLS outputs:", [c.shape for c in cls_out])
print("REG outputs:", [r.shape for r in reg_out])
del model; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

CLS outputs: [torch.Size([2, 135, 80, 80]), torch.Size([2, 135, 40, 40]), torch.Size([2, 135, 20, 20])]
REG outputs: [torch.Size([2, 63, 80, 80]), torch.Size([2, 63, 40, 40]), torch.Size([2, 63, 20, 20])]


In [ ]:
import cv2
import torchvision.transforms as T

In [ ]:
import torch.optim as optim
from collections import defaultdict

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = '/content/drive/MyDrive/msml605_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)

ARGOVERSE_CLASSES = [
    "VEHICLE", "PEDESTRIAN", "ON_ROAD_OBSTACLE", "LARGE_VEHICLE", "BICYCLE",
    "BICYCLIST", "BUS", "OTHER_MOVER", "TRAILER", "MOTORCYCLIST", "MOPED",
    "MOTORCYCLE", "STROLLER", "EMERGENCY_VEHICLE", "ANIMAL"
]
CLASS_TO_IDX = {c: i for i, c in enumerate(ARGOVERSE_CLASSES)}

NUM_CLASSES = 15
NUM_ANCHORS = 9

def build_cls_targets(cls_out, batch_annots):
    """
    Build real classification targets from ground-truth annotations.
    For each GT object, finds its rough grid cell and marks that
    cell positive for the correct class across all anchors.

    cls_out: tensor of shape [B, NUM_ANCHORS*NUM_CLASSES, H, W]
    batch_annots: list of length B, each a list of annotation dicts
    """
    B, _, H, W = cls_out.shape
    target = torch.zeros(B, NUM_ANCHORS * NUM_CLASSES, H, W, device=cls_out.device)

    for b, annots in enumerate(batch_annots):
        for obj in annots:
            label = obj.get("label", "")
            if label not in CLASS_TO_IDX:
                continue
            cls_idx = CLASS_TO_IDX[label]

            # Argoverse center is in vehicle/city frame (metres).
            # Map x in [-50, 50] → [0, W], y in [-50, 50] → [0, H]
            cx = obj["center"].get("x", 0.0)
            cy = obj["center"].get("y", 0.0)
            gi = int(np.clip((cx + 50.0) / 100.0 * W, 0, W - 1))
            gj = int(np.clip((cy + 50.0) / 100.0 * H, 0, H - 1))

            # Mark all anchors positive at this cell for this class
            for a in range(NUM_ANCHORS):
                target[b, a * NUM_CLASSES + cls_idx, gj, gi] = 1.0

    return target


def train_one_config(depth, width_mult, train_loader, num_epochs=8):
    ckpt_path = f"{CKPT_DIR}resnet{depth}_w{width_mult}.pth"


    model     = Detector3D(depth=depth, width_mult=width_mult, num_classes=NUM_CLASSES).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # pos_weight downweights the overwhelming number of background cells
    pos_weight = torch.tensor([10.0]).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_loader:
            images       = torch.stack([b[0] for b in batch]).to(device)
            batch_annots = [b[1] for b in batch]   # list of annotation lists

            optimizer.zero_grad()
            cls_outs, _ = model(images)

            # Build real targets for each FPN level and compute loss
            loss = sum(
                criterion(c, build_cls_targets(c, batch_annots))
                for c in cls_outs
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()

        print(f"  Epoch {epoch+1}/{num_epochs}  loss: {total_loss/len(train_loader):.4f}")

    torch.save(model.state_dict(), ckpt_path)
    print(f"  Saved → {ckpt_path}")
    del model; torch.cuda.empty_cache()
    return ckpt_path


for depth in [101]:
    for wm in [0.5]:
        print(f"\nTraining ResNet-{depth} width={wm}x ...")
        train_one_config(depth, wm, train_loader, num_epochs=8)


Training ResNet-101 width=0.5x ...


model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

  Epoch 1/8  loss: 0.2945
  Epoch 2/8  loss: 0.1405
  Epoch 3/8  loss: 0.1257
  Epoch 4/8  loss: 0.1131
  Epoch 5/8  loss: 0.1016
  Epoch 6/8  loss: 0.0902
  Epoch 7/8  loss: 0.0801
  Epoch 8/8  loss: 0.0710
  Saved → /content/drive/MyDrive/msml605_checkpoints/resnet101_w0.5.pth


In [ ]:
from fvcore.nn import FlopCountAnalysis, parameter_count

dummy_input = torch.randn(1, 3, 640, 640).cuda()
results = []

for depth in [18, 34, 50, 101]:
    for wm in [0.5, 1.0, 2.0]:
        print(f"Running ResNet-{depth} width={wm}x ...")
        try:
            torch.cuda.empty_cache()
            baseline_mem_mb = torch.cuda.memory_allocated() / 1e6
            model = Detector3D(depth=depth, width_mult=wm, num_classes=15).cuda().eval()

            flops = FlopCountAnalysis(model, dummy_input)
            flops.unsupported_ops_warnings(False)
            total_flops = flops.total()
            params      = parameter_count(model)['']

            torch.cuda.reset_peak_memory_stats()
            with torch.no_grad(): model(dummy_input)
            torch.cuda.synchronize()
            peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) - baseline_mem_mb

            results.append({"depth": depth, "width_mult": wm,
                            "GFLOPs": round(total_flops/1e9, 3),
                            "params_M": round(params/1e6, 3),
                            "peak_mem_MB": round(peak_mem_mb, 2)})
            print(f"  GFLOPs: {total_flops/1e9:.2f} | Params: {params/1e6:.2f}M | Mem: {peak_mem_mb:.1f}MB")
        except Exception as e:
            print(f"  ERROR: {e}")
        finally:
            del model; torch.cuda.empty_cache()

df_flops = pd.DataFrame(results)
print(df_flops)
df_flops.to_csv(f"{CKPT_DIR}flop_counts.csv", index=False)

Running ResNet-18 width=0.5x ...
  GFLOPs: 18.96 | Params: 12.06M | Mem: 48.5MB
Running ResNet-18 width=1.0x ...
  GFLOPs: 30.50 | Params: 14.41M | Mem: 65.3MB
Running ResNet-18 width=2.0x ...
  GFLOPs: 75.89 | Params: 23.54M | Mem: 103.0MB
Running ResNet-34 width=0.5x ...
  GFLOPs: 34.08 | Params: 22.16M | Mem: 50.2MB
Running ResNet-34 width=1.0x ...
  GFLOPs: 45.62 | Params: 24.52M | Mem: 62.8MB
Running ResNet-34 width=2.0x ...
  GFLOPs: 91.01 | Params: 33.65M | Mem: 105.8MB
Running ResNet-50 width=0.5x ...
  GFLOPs: 38.21 | Params: 24.73M | Mem: 53.4MB
Running ResNet-50 width=1.0x ...
  GFLOPs: 50.31 | Params: 27.43M | Mem: 104.2MB
Running ResNet-50 width=2.0x ...


model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

  GFLOPs: 156.51 | Params: 80.57M | Mem: 339.9MB
Running ResNet-101 width=0.5x ...
  GFLOPs: 68.60 | Params: 43.72M | Mem: -55.1MB
Running ResNet-101 width=1.0x ...
  GFLOPs: 80.69 | Params: 46.42M | Mem: 101.7MB
Running ResNet-101 width=2.0x ...


model.safetensors:   0%|          | 0.00/508M [00:00<?, ?B/s]

  GFLOPs: 249.31 | Params: 138.57M | Mem: 501.9MB
    depth  width_mult   GFLOPs  params_M  peak_mem_MB
0      18         0.5   18.958    12.055        48.46
1      18         1.0   30.504    14.408        65.28
2      18         2.0   75.889    23.537       103.03
3      34         0.5   34.078    22.163        50.22
4      34         1.0   45.624    24.516        62.79
5      34         2.0   91.009    33.646       105.79
6      50         0.5   38.210    24.731        53.39
7      50         1.0   50.305    27.428       104.21
8      50         2.0  156.509    80.571       339.94
9     101         0.5   68.597    43.723       -55.10
10    101         1.0   80.693    46.420       101.65
11    101         2.0  249.315   138.575       501.89


In [ ]:
WARMUP  = 50
MEASURE = 200
dummy_input    = torch.randn(1, 3, 640, 640).cuda()
timing_results = []

for depth in [18, 34, 50, 101]:
    for wm in [0.5, 1.0, 2.0]:
        print(f"Timing ResNet-{depth} width={wm}x ...")
        model = Detector3D(depth=depth, width_mult=wm, num_classes=15).cuda().eval()

        with torch.no_grad():
            for _ in range(WARMUP): model(dummy_input)

        times = []
        with torch.no_grad():
            for _ in range(MEASURE):
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                start.record(); model(dummy_input); end.record()
                torch.cuda.synchronize()
                times.append(start.elapsed_time(end))

        mean_ms, std_ms = np.mean(times), np.std(times)
        print(f"  Mean: {mean_ms:.2f}ms | Std: {std_ms:.2f}ms")
        timing_results.append({"depth": depth, "width_mult": wm,
                               "mean_ms": round(mean_ms, 3), "std_ms": round(std_ms, 3)})
        del model; torch.cuda.empty_cache()

df_timing = pd.DataFrame(timing_results)
df_full   = pd.merge(df_flops, df_timing, on=["depth", "width_mult"])
print(df_full)
df_full.to_csv(f"{CKPT_DIR}timing_results.csv", index=False)

In [ ]:
dummy_input      = torch.randn(1, 3, 640, 640).cuda()
profiler_results = {}

for depth in [18, 50, 101]:
    for wm in [1.0]:
        config_name = f"ResNet-{depth}-w{wm}"
        print(f"\nProfiling {config_name} ...")
        model = Detector3D(depth=depth, width_mult=wm, num_classes=15).cuda().eval()

        with torch.no_grad():
            for _ in range(10): model(dummy_input)

        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                     record_shapes=True, profile_memory=True, with_flops=True) as prof:
            with record_function("model_inference"):
                with torch.no_grad():
                    for _ in range(20): model(dummy_input)

        print(prof.key_averages().table(sort_by="self_device_time_total", row_limit=10))

        rows = [{"config": config_name, "op_name": evt.key,
                 "cpu_time_us": evt.cpu_time_total,
                 "cuda_time_us": evt.self_device_time_total,
                 "cuda_mem_bytes": evt.device_memory_usage,
                 "flops": getattr(evt, "flops", 0)}
                for evt in prof.key_averages()]
        profiler_results[config_name] = rows
        del model; torch.cuda.empty_cache()

df_prof = pd.DataFrame([r for rows in profiler_results.values() for r in rows])
df_prof.to_csv(f"{CKPT_DIR}profiler_results.csv", index=False)
print("Profiler results saved.")

In [ ]:
import subprocess

def get_nvidia_smi_memory():
    result = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                             "--format=csv,noheader,nounits"],
                            capture_output=True, text=True)
    if result.returncode != 0: return None, None
    used, total = [int(x.strip()) for x in result.stdout.strip().split("\n")[0].split(",")]
    return used, total

dummy_input      = torch.randn(1, 3, 640, 640).cuda()
nvidia_smi_results = []

for depth in [18, 34, 50, 101]:
    for wm in [0.5, 1.0, 2.0]:
        torch.cuda.empty_cache()
        model = Detector3D(depth=depth, width_mult=wm, num_classes=15).cuda().eval()
        with torch.no_grad(): model(dummy_input)
        torch.cuda.synchronize()
        smi_used, smi_total = get_nvidia_smi_memory()
        nvidia_smi_results.append({"depth": depth, "width_mult": wm,
                                   "smi_used_MB": smi_used, "smi_total_MB": smi_total,
                                   "smi_free_MB": (smi_total - smi_used) if smi_total else None})
        print(f"ResNet-{depth} w{wm}x | {smi_used}/{smi_total} MB")
        del model; torch.cuda.empty_cache()

df_smi = pd.DataFrame(nvidia_smi_results)
df_smi.to_csv(f"{CKPT_DIR}nvidia_smi_memory.csv", index=False)

In [ ]:
T4_PEAK_TFLOPS_FP32 = 8.1
T4_PEAK_BW_GB_S     = 300.0
RIDGE_POINT         = (T4_PEAK_TFLOPS_FP32 * 1e12) / (T4_PEAK_BW_GB_S * 1e9)

df_roofline = df_full.copy()
df_roofline["bytes_moved"]      = df_roofline["peak_mem_MB"] * 1e6
df_roofline["FLOPs"]            = df_roofline["GFLOPs"] * 1e9
df_roofline["arith_intensity"]  = df_roofline["FLOPs"] / df_roofline["bytes_moved"]
df_roofline["actual_TFLOPS"]    = (df_roofline["FLOPs"] / (df_roofline["mean_ms"] * 1e-3)) / 1e12
df_roofline["bound"]            = df_roofline["arith_intensity"].apply(
    lambda ai: "Memory-bound" if ai < RIDGE_POINT else "Compute-bound")

print(df_roofline[["depth","width_mult","arith_intensity","actual_TFLOPS","bound"]].to_string())
df_roofline.to_csv(f"{CKPT_DIR}roofline_analysis.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
ai_range = np.logspace(-1, 4, 500)
ax.loglog(ai_range, np.minimum(T4_PEAK_TFLOPS_FP32, (T4_PEAK_BW_GB_S * ai_range) / 1e3),
          'k-', linewidth=2, label="Roofline (T4)")
ax.axvline(RIDGE_POINT, color='gray', linestyle='--', alpha=0.6,
           label=f"Ridge ({RIDGE_POINT:.0f} FLOPs/B)")

colors  = {18: 'tab:blue', 34: 'tab:orange', 50: 'tab:green', 101: 'tab:red'}
markers = {0.5: 's', 1.0: 'o', 2.0: '^'}
for _, row in df_roofline.iterrows():
    ax.scatter(row["arith_intensity"], row["actual_TFLOPS"],
               color=colors[int(row["depth"])], marker=markers[row["width_mult"]], s=120, zorder=5)
    ax.annotate(f"R{int(row['depth'])}\n{row['width_mult']}x",
                (row["arith_intensity"], row["actual_TFLOPS"]),
                textcoords="offset points", xytext=(6, 4), fontsize=7)

ax.set_xlabel("Arithmetic Intensity (FLOPs / Byte)", fontsize=12)
ax.set_ylabel("Throughput (TFLOPS)", fontsize=12)
ax.set_title("Roofline Model: ResNet Scaling on T4 GPU", fontsize=13)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CKPT_DIR}roofline_plot.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
for wm, grp in df_full.groupby("width_mult"):
    ax.plot(grp["GFLOPs"], grp["mean_ms"], marker='o', label=f"width={wm}x")
slope, intercept, r, *_ = stats.linregress(df_full["GFLOPs"], df_full["mean_ms"])
x_fit = np.linspace(df_full["GFLOPs"].min(), df_full["GFLOPs"].max(), 100)
ax.plot(x_fit, slope*x_fit + intercept, 'k--', alpha=0.5, label=f"Linear fit (R²={r**2:.2f})")
ax.set_xlabel("GFLOPs"); ax.set_ylabel("Mean Inference Time (ms)")
ax.set_title("FLOPs vs Runtime"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for wm, grp in df_full.groupby("width_mult"):
    ax.plot(grp["depth"], grp["mean_ms"], marker='o', label=f"width={wm}x")
ax.set_xlabel("ResNet Depth"); ax.set_ylabel("Mean Inference Time (ms)")
ax.set_title("Depth vs Runtime"); ax.set_xticks([18, 34, 50, 101])
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
for wm, grp in df_full.groupby("width_mult"):
    ax.plot(grp["depth"], grp["peak_mem_MB"], marker='s', label=f"width={wm}x")
ax.set_xlabel("ResNet Width"); ax.set_ylabel("Peak Memory (MB)")
ax.set_title("Memory vs Width"); ax.set_xticks([18, 34, 50, 101])
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("ResNet Scaling Analysis — T4 GPU", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{CKPT_DIR}scaling_plots.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from collections import defaultdict

ARGOVERSE_CLASSES = [
    "VEHICLE", "PEDESTRIAN", "ON_ROAD_OBSTACLE", "LARGE_VEHICLE", "BICYCLE",
    "BICYCLIST", "BUS", "OTHER_MOVER", "TRAILER", "MOTORCYCLIST", "MOPED",
    "MOTORCYCLE", "STROLLER", "EMERGENCY_VEHICLE", "ANIMAL"
]
CLASS_TO_IDX = {c: i for i, c in enumerate(ARGOVERSE_CLASSES)}
NUM_CLASSES  = 15
NUM_ANCHORS  = 9
SCORE_THRESH = 0.3   


def compute_ap(recalls, precisions):
    ap = 0.0
    for t in np.arange(0.0, 1.1, 0.1):
        prec_at_rec = [p for r, p in zip(recalls, precisions) if r >= t]
        ap += max(prec_at_rec) if prec_at_rec else 0.0
    return ap / 11.0


def evaluate_map(dataset, model, device, max_samples=100):
    model.eval()
    all_detections = defaultdict(list)
    all_gt_counts  = defaultdict(int)
    transform      = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    with torch.no_grad():
        for i in range(min(max_samples, len(dataset))):
            image, annots, seq, ts = dataset[i]
            if not annots:
                continue

            
            for gt in annots:
                if gt["label"] in CLASS_TO_IDX:
                    all_gt_counts[gt["label"]] += 1

            img_tensor  = transform(image).unsqueeze(0).to(device)
            cls_outs, _ = model(img_tensor)

            
            raw = cls_outs[0][0]           # shape: [NUM_ANCHORS*NUM_CLASSES, H, W]
            H, W = raw.shape[1], raw.shape[2]

           
            scores = torch.sigmoid(raw).reshape(NUM_ANCHORS, NUM_CLASSES, H, W)

            
            gt_labels_in_scene = {gt["label"] for gt in annots if gt["label"] in CLASS_TO_IDX}

            for cls_i in range(NUM_CLASSES):
                cls_scores = scores[:, cls_i, :, :]   # [anchors, H, W]
                
                best_scores, _ = cls_scores.max(dim=0)  # [H, W]
                above_thresh   = (best_scores > SCORE_THRESH).nonzero(as_tuple=False)

                for pos in above_thresh:
                    score    = best_scores[pos[0], pos[1]].item()
                    pred_cls = ARGOVERSE_CLASSES[cls_i]
                    is_tp    = int(pred_cls in gt_labels_in_scene)
                    all_detections[pred_cls].append((score, is_tp))

    aps = {}
    for cls_name in ARGOVERSE_CLASSES:
        dets = sorted(all_detections.get(cls_name, []), key=lambda x: -x[0])
        if not dets or all_gt_counts.get(cls_name, 0) == 0:
            continue
        tp_cum, fp_cum, recalls, precisions = 0, 0, [], []
        for score, is_tp in dets:
            if is_tp: tp_cum += 1
            else:     fp_cum += 1
            recalls.append(tp_cum / all_gt_counts[cls_name])
            precisions.append(tp_cum / (tp_cum + fp_cum))
        aps[cls_name] = compute_ap(recalls, precisions)

    return np.mean(list(aps.values())) if aps else 0.0, aps


map_results = []
for depth in [18, 34, 50, 101]:
    for wm in [0.5, 1.0, 2.0]:
        print(f"\nEvaluating mAP: ResNet-{depth} width={wm}x ...")
        model     = Detector3D(depth=depth, width_mult=wm, num_classes=NUM_CLASSES).to(device)
        ckpt_path = f"{CKPT_DIR}resnet{depth}_w{wm}.pth"
        if os.path.exists(ckpt_path):
            model.load_state_dict(torch.load(ckpt_path, map_location=device))
            print(f"  Loaded checkpoint")
        else:
            print(f"  No checkpoint — random weights")
        mAP, per_class = evaluate_map(val_dataset, model, device, max_samples=100)
        print(f"  mAP: {mAP:.4f}")
        if per_class:
            for cls, ap in sorted(per_class.items(), key=lambda x: -x[1]):
                print(f"    {cls}: {ap:.4f}")
        map_results.append({"depth": depth, "width_mult": wm, "mAP": round(mAP, 4)})
        del model; torch.cuda.empty_cache()

df_map = pd.DataFrame(map_results)
print("\n", df_map)
df_map.to_csv(f"{CKPT_DIR}map_results.csv", index=False)
print(f"Saved → {CKPT_DIR}map_results.csv")

In [ ]:
import os
import shutil

CKPT_DIR  = '/content/drive/MyDrive/msml605_checkpoints/'
SAVE_DIR  = '/content/drive/MyDrive/msml605_results/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save all DataFrames
df_flops.to_csv(f"{SAVE_DIR}flop_counts.csv", index=False)
df_timing.to_csv(f"{SAVE_DIR}timing_results.csv", index=False)
df_full.to_csv(f"{SAVE_DIR}full_results.csv", index=False)
df_roofline.to_csv(f"{SAVE_DIR}roofline_analysis.csv", index=False)
df_map.to_csv(f"{SAVE_DIR}map_results.csv", index=False)
df_smi.to_csv(f"{SAVE_DIR}nvidia_smi_memory.csv", index=False)
df_prof.to_csv(f"{SAVE_DIR}profiler_results.csv", index=False)

# Save all plots
plt.figure(figsize=(16, 5))
for wm, grp in df_full.groupby("width_mult"):
    plt.plot(grp["GFLOPs"], grp["mean_ms"], marker='o', label=f"width={wm}x")
plt.xlabel("GFLOPs"); plt.ylabel("Mean Inference Time (ms)")
plt.title("FLOPs vs Runtime"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{SAVE_DIR}flops_vs_runtime.png", dpi=150, bbox_inches='tight')
plt.close()

# Roofline plot
from scipy import stats
import numpy as np
T4_PEAK_TFLOPS_FP32 = 8.1
T4_PEAK_BW_GB_S     = 300.0
RIDGE_POINT         = (T4_PEAK_TFLOPS_FP32 * 1e12) / (T4_PEAK_BW_GB_S * 1e9)
fig, ax = plt.subplots(figsize=(10, 6))
ai_range = np.logspace(-1, 4, 500)
ax.loglog(ai_range, np.minimum(T4_PEAK_TFLOPS_FP32, (T4_PEAK_BW_GB_S * ai_range) / 1e3),
          'k-', linewidth=2, label="Roofline (T4)")
ax.axvline(RIDGE_POINT, color='gray', linestyle='--', alpha=0.6)
colors  = {18: 'tab:blue', 34: 'tab:orange', 50: 'tab:green', 101: 'tab:red'}
markers = {0.5: 's', 1.0: 'o', 2.0: '^'}
for _, row in df_roofline.iterrows():
    ax.scatter(row["arith_intensity"], row["actual_TFLOPS"],
               color=colors[int(row["depth"])], marker=markers[row["width_mult"]], s=120, zorder=5)
    ax.annotate(f"R{int(row['depth'])}\n{row['width_mult']}x",
                (row["arith_intensity"], row["actual_TFLOPS"]),
                textcoords="offset points", xytext=(6, 4), fontsize=7)
ax.set_xlabel("Arithmetic Intensity (FLOPs/Byte)")
ax.set_ylabel("Throughput (TFLOPS)")
ax.set_title("Roofline — T4 GPU")
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}roofline_plot.png", dpi=150)
plt.close()

print("✅ All results saved to Drive:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {SAVE_DIR}{f}")